# Why Logistic Regression: Linear Failures & The Sigmoid Curve Lab

Linear regression fits continuous targets ($y \in \mathbb{R}$), but applying it to binary classification ($y \in \{0, 1\}$) leads to two critical failures: impossible probabilities outside $[0, 1]$ and catastrophic line tilts caused by outliers. This lab demonstrates these limitations and shows why the sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$ provides the proper mathematical foundation for classification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit  # Scipy implementation of sigmoid
from sklearn.linear_model import LinearRegression, LogisticRegression

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Failure Mode 1: Impossible Probabilities with Linear Regression

Fit Ordinary Least Squares linear regression on student study hours vs. pass/fail outcomes. Observe how predictions escape $[0, 1]$ on extreme hours.

In [ ]:
hours = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 11]).reshape(-1, 1)
passed = np.array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1])

lin_reg = LinearRegression().fit(hours, passed)
preds_linear = lin_reg.predict(hours)

print(f"Linear Equation: y = {lin_reg.coef_[0]:.3f}x + {lin_reg.intercept_:.3f}")
print(f"{'Hours':<6} {'Actual':<8} {'Linear Pred':<14} {'Anomaly Status'}")
print("-" * 46)
for h, y_true, pred in zip(hours.ravel(), passed, preds_linear):
    anomaly = "" if 0.0 <= pred <= 1.0 else ("NEGATIVE PROBABILITY!" if pred < 0 else "PROBABILITY > 100%!")
    print(f"{h:<6} {y_true:<8} {pred:<14.3f} {anomaly}")

## 2. Failure Mode 2: Outlier Vulnerability & Decision Boundary Shift

Add a single student who studied 50 hours but failed due to an anomaly. Notice how OLS tilts the regression line to minimize squared error, destroying accuracy on borderline students.

In [ ]:
# Add outlier point: (50 hours, 0 failed)
hours_corrupt = np.vstack([hours, [[50]]])
passed_corrupt = np.append(passed, 0)

lin_corrupt = LinearRegression().fit(hours_corrupt, passed_corrupt)
log_corrupt = LogisticRegression().fit(hours_corrupt, passed_corrupt)

print(f"Clean Linear Slope:    {lin_reg.coef_[0]:.4f}")
print(f"Corrupted Linear Slope:{lin_corrupt.coef_[0]:.4f} (Severe slope dampening!)")

# Check prediction on student who studied 6 hours
print(f"\nPrediction for student with 6 hours studied:")
print(f"  Clean Linear Fit:     {lin_reg.predict([[6]])[0]:.3f} (Predicted PASS at threshold 0.5)")
print(f"  Outlier Linear Fit:   {lin_corrupt.predict([[6]])[0]:.3f} (Flipped to FAIL!)")
print(f"  Logistic Probability: {log_corrupt.predict_proba([[6]])[0][1]:.3f} (Still safely PASS!)")

## 3. The Sigmoid Function & Mathematical Symmetry

Inspect the properties of $\sigma(z) = \frac{1}{1 + e^{-z}}$, verifying the essential symmetry property $\sigma(-z) = 1 - \sigma(z)$.

In [ ]:
z_points = np.array([-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0])
sig_vals = expit(z_points)

print(f"{'z':<6} {'σ(z)':<10} {'1 - σ(-z)':<12} {'Symmetry Match'}")
print("-" * 40)
for z, s in zip(z_points, sig_vals):
    sym = 1.0 - expit(-z)
    match = np.isclose(s, sym)
    print(f"{z:<6.1f} {s:<10.4f} {sym:<12.4f} {match}")